# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [338]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier


In [320]:
spaceship_original = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship = spaceship_original.copy()
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [321]:
spaceship.shape

spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [322]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [323]:
# for null values in numeric variables, impute with median 

def fill_null(col):
    if pd.api.types.is_numeric_dtype(col):
        return col.fillna(col.median())
    return col
    

spaceship = spaceship.apply(fill_null)


In [324]:
# for null values in categorical variables, impute with mode

def full_na_catogrical(col):
    if pd.api.types.is_object_dtype(col):
        return col.fillna(col.mode()[0])
    return col

spaceship = spaceship.apply(full_na_catogrical)



/var/folders/zd/t_pj2nzd2_x4z46k814jg73r0000gn/T/ipykernel_24929/4211567268.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return col.fillna(col.mode()[0])


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [325]:
s = spaceship['Cabin'].str.split('/') # a series, each element is a list

df = s.apply(pd.Series) # mini dataframe object

df.rename(columns={0:'Cabin_deck',1:'Cabin_num',2:'Cabin_side'}, inplace=True)

spaceship = pd.concat([spaceship,df],axis=1)



In [326]:
spaceship.head(3)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Cabin_deck,Cabin_num,Cabin_side
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B,0,P
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F,0,S
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A,0,S


- Drop PassengerId and Name

In [327]:
spaceship.drop(columns=['PassengerId','Cabin','Name','Cabin_side','Cabin_num','Destination'],axis= 0, inplace=True)


In [328]:
spaceship

,HomePlanet,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Cabin_deck
0,Europa,False,39.0,False,0.0,0.0,0.0,0.0,0.0,False,B
1,Earth,False,24.0,False,109.0,9.0,25.0,549.0,44.0,True,F
2,Europa,False,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,A
3,Europa,False,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,A
4,Earth,False,16.0,False,303.0,70.0,151.0,565.0,2.0,True,F
...,...,...,...,...,...,...,...,...,...,...,...
8688,Europa,False,41.0,True,0.0,6819.0,0.0,1643.0,74.0,False,A
8689,Earth,True,18.0,False,0.0,0.0,0.0,0.0,0.0,False,G
8690,Earth,False,26.0,False,0.0,0.0,1872.0,1.0,0.0,True,G
8691,Europa,False,32.0,False,0.0,1049.0,0.0,353.0,3235.0,False,E


**Perform Train Test Split**

In [329]:
X = spaceship.drop(columns=['Transported'])

y = spaceship['Transported']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_test


,HomePlanet,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Cabin_deck
304,Mars,False,19.0,False,417.0,349.0,634.0,3.0,1057.0,F
2697,Earth,False,18.0,False,4.0,904.0,0.0,0.0,1.0,G
8424,Earth,True,41.0,False,0.0,0.0,0.0,0.0,0.0,G
1672,Earth,False,35.0,False,0.0,338.0,436.0,0.0,0.0,G
8458,Europa,True,43.0,False,0.0,0.0,0.0,0.0,0.0,D
...,...,...,...,...,...,...,...,...,...,...
7175,Earth,True,16.0,False,0.0,0.0,0.0,0.0,0.0,G
3187,Earth,True,0.0,False,0.0,0.0,0.0,0.0,0.0,G
1302,Earth,False,17.0,False,21.0,0.0,690.0,260.0,5.0,E
5934,Mars,True,42.0,False,0.0,0.0,0.0,0.0,0.0,F


#### For non-numerical columns, do dummies.

In [ ]:
# identify categorical and numeric variables 
cat_cols = ["HomePlanet", "CryoSleep", "VIP", "Cabin_deck"]
numeric_cols = [col for col in X_train.columns if col not in cat_cols]


# create ColumnTransformer object from ColumnTransformer class 
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

# Fit on train set
preprocessor.fit(X_train)

# Transform train and test
X_train_enc = preprocessor.transform(X_train)
X_test_enc = preprocessor.transform(X_test)

# Get feature names
feature_names = (
    preprocessor.named_transformers_["cat"].get_feature_names_out(cat_cols).tolist()
    + numeric_cols
)

# Convert to DataFrame
X_train_df = pd.DataFrame(X_train_enc, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_enc, columns=feature_names, index=X_test.index)



**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [362]:
knn = KNeighborsClassifier(n_neighbors=20)

knn.fit(X_train_df,y_train)

predict = knn.predict(X_test_df)

predict

array([False,  True,  True, ..., False,  True, False])

- Evaluate your model's performance. Comment it

In [ ]:
from sklearn.metrics import precision_score, recall_score


# Precision, of all predicted positives, how many are actually positive.
# Out of all passengers predicted as Transported, ~78% were actually transported.
# So, when the model says “yes”, it’s correct 78% of the time.

# Recall, of all actual positives, how many did we correctly find.
# Out of all passengers who were actually Transported, the model correctly caught ~77.6%.
# So, it misses about 22% of the transported passengers.

precision_test = precision_score(y_test, predict)
recall_test = recall_score(y_test, predict)

pred_train = knn.predict(X_train_df)

precision_train = precision_score(y_train, pred_train)
recall_train = recall_score(y_train, pred_train)

# check overfitting 
print("test Precision:", precision_test)
print("test Recall:", recall_test)

print("train Precision:", precision_train)
print("train Recall:", recall_train)



test Precision: 0.7742290748898678
test Recall: 0.8006833712984055
train Precision: 0.7976091187100361
train Recall: 0.8197142857142857
